In [1]:
# 다중분류... 뉴스 카테고리
# DATA_URL = "https://raw.githubusercontent.com/kocohub/korean-hate-speech/master/labeled/train.tsv"

In [2]:
import urllib
url = "https://raw.githubusercontent.com/kocohub/korean-hate-speech/master/labeled/train.tsv"
def download_data():
    urllib.request.urlretrieve(url, 'train_hate_tsv')

download_data()

In [3]:
import pandas as pd
df = pd.read_csv('train_hate_tsv', sep='\t').dropna()
# None 없음 hate 싫음 offensive 공격적
df.head()

,comments,contain_gender_bias,bias,hate
0,(현재 호텔주인 심정) 아18 난 마른하늘에 날벼락맞고 호텔망하게생겼는데 누군 계속...,False,others,hate
1,....한국적인 미인의 대표적인 분...너무나 곱고아름다운모습...그모습뒤의 슬픔을...,False,none,none
2,"...못된 넘들...남의 고통을 즐겼던 넘들..이젠 마땅한 처벌을 받아야지..,그래...",False,none,hate
3,"1,2화 어설펐는데 3,4화 지나서부터는 갈수록 너무 재밌던데",False,none,none
4,1. 사람 얼굴 손톱으로 긁은것은 인격살해이고2. 동영상이 몰카냐? 메걸리안들 생각...,True,gender,hate


In [10]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['hate'])
label_encoder.classes_

array(['hate', 'none', 'offensive'], dtype=object)

In [12]:
#학습/테스트데이터 분리(8:2)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df,test_size=0.2,random_state=42)
train_df.shape, test_df.shape, df.shape

((6316, 5), (1580, 5), (7896, 5))

In [19]:
# 전처리 및 어휘사전(vocab) 구축
from konlpy.tag import Okt
okt = Okt()
# okt.pos('아버지가 방에 들어가신다'), okt.morphs('아버지가 방에 들어가신다'), okt.phrases('아버지가 방에 들어가신다')
def konlpy_tokenizer(text):
    return okt.morphs(text)

train_tokenized = []
for text in train_df['comments']:
    tokens = konlpy_tokenizer(text)
    train_tokenized.append(tokens)

In [33]:
from collections import Counter
#빈도수 기준 단어사전 생성
vocab_size = 8000
all_tokens = [ token for token_list in train_tokenized  for token in token_list  ]
word_counts = Counter(all_tokens)
vocab = {'<PAD>':0, '<UNK>':1}

for word in set([word for word,cnt in word_counts.most_common(vocab_size-2)]):
    vocab[word] = len(vocab)

# 문장의 길이를 통일  시퀀스
from torch.nn.utils.rnn import pad_sequence
import torch
sequence_len = 30
def text_to_sequence(tokens, max_len=30):
    seq = [vocab.get(token,1) for token in tokens]
    seq = pad_sequence( torch.LongTensor(seq[:max_len]))

[text_to_sequence(tokens) for tokens in train_tokenized    ]

RuntimeError: ArrayRef: invalid slice, N = 1; size = 0

In [30]:
train_tokenized

[['ㅇ',
  '정보',
  '부',
  '에서',
  '조',
  '사하나',
  '봐요',
  '..',
  '승리',
  '곧',
  '자살',
  '하겠네',
  '아니',
  '..',
  '경찰',
  '이',
  '질소',
  '넣고',
  '자살',
  '했다고',
  '하겠네',
  '..'],
 ['어차피', '속', '바지', '인데', '뭔', '노출', 'ㅋㅋ'],
 ['손담비', '나혼', '에', '고정', '으로', '가자', '~~', '매력', '넘침', '~'],
 ['개', '듣보네', 'ㅋㅋㅋㅋㅋㅋ', '남자', '뭔', '프로게이머', '같이', '생겼네'],
 ['누가', '봐도', '한', '거', '다', '.', 'ㅎㅎㅎ'],
 ['처음',
  '본',
  '여자',
  '한테',
  '찝',
  '적',
  '거렸는데',
  ',',
  '여자',
  '가',
  '기분',
  '나빳다',
  '?',
  '이',
  '게',
  '죄',
  '야',
  '?',
  '성적',
  '발언',
  '을',
  '한',
  '것',
  '도',
  '아니고',
  ',',
  '상대',
  '가',
  '기분',
  '이',
  '나쁘면',
  '무조건',
  '피해자',
  '로',
  '규정',
  '하나',
  '?',
  '이건',
  '아니지',
  '싶다',
  'ㅋㅋㅋ'],
 ['에이', '핑', '큰줄', '알았더니', '짝퉁', '김남주', '였네'],
 ['개',
  '쳐',
  '늙으면',
  '꼴',
  '보기',
  '싫어지는게',
  '사람',
  '의',
  '마음',
  '이지',
  '얘',
  '네',
  '야',
  '광고',
  '수익',
  '으로',
  '번',
  '게',
  '많으니',
  '얼굴',
  '에',
  '돈',
  '들',
  '이고',
  '관리',
  '하니',
  '금술',
  '이',
  '당연히',
  '좋을수밖에',
  '사람',
  '이란',
 